# See-through 전용 Live2D PSD 제작: 자산별 원화 ZIP 합성

이 노트북은 GPT·Gemini가 만든 **완성 원화 ZIP** 3개(FACE, OUTFIT, HAIR)와 선택적 ACCESSORY 1개를 입력받습니다.

1. 먼저 자산별 원화를 **같은 캔버스·정면 구도·얼굴/목/옷깃 좌표**에 맞춰 투명 RGBA로 생성합니다. 서로 다른 캔버스는 거부합니다.
2. 합성 미리보기를 **직접 육안으로 검수**합니다. 이 검사는 시각적 일치를 자동 증명하지 않습니다.
3. 입력 ZIP들이 모두 `part_art/_order.txt` 및 독립 RGBA 레이어를 포함하면 **원본 파츠를 보존해 PSD로 직접 조립**하고 GPU 분리를 생략합니다. 독립 파츠가 없는 경우에만 See-through (LayerDiff/Marigold + 공식 SemanticSAM 후처리)를 사용합니다. Qwen/Stable-Layers는 사용하지 않습니다.
4. 기본 출력은 파츠 PSD+PNG ZIP입니다. 선택적 PSD2Live 단계를 실행하면 MOC3/CMO3 제작을 시도하지만 실제 방송 동작은 별도 검증해야 합니다.

**주의:** FACE 단독, 헤어 단독, 의상 단독으로 See-through를 실행하지 마세요. 합성 완료된 캐릭터 정면 원화에서만 분리합니다.

In [ ]:
#@title ① 저장소 main 동기화
from pathlib import Path
import subprocess, sys, uuid, json
REPO = Path("/content/Virtual-pipeline")
URL = "https://github.com/jujumelona/Virtual-pipeline.git"
if (REPO / ".git").is_dir():
    subprocess.run(["git","-C",str(REPO),"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-B","main","origin/main"],check=True)
else:
    if REPO.exists():
        REPO = Path("/content/Virtual-pipeline-"+uuid.uuid4().hex[:8])
    subprocess.run(["git","clone","--depth","1","--branch","main",URL,str(REPO)],check=True)
sys.path.insert(0,str(REPO))
from tools.vts_see_through_only import composite_2d, run_see_through
print("See-through 전용 코드 준비:", REPO)

In [ ]:
#@title ② 원화 ZIP 3개 및 선택적 ACCESSORY 업로드
from google.colab import files
WORK = Path("/content/see_through_source_jobs") / uuid.uuid4().hex
WORK.mkdir(parents=True)
WITH_ACCESSORY = False #@param {type:"boolean"}
def upload_asset(name):
    print(name, "ARTWORK ZIP 파일을 업로드하세요")
    selected = files.upload()
    if len(selected) != 1:
        raise ValueError(name+": ZIP 하나만 업로드하세요")
    filename, raw = next(iter(selected.items()))
    if not filename.lower().endswith(".zip") or not raw:
        raise ValueError(name+": 완성 원화 ZIP이 아닙니다")
    target = WORK / (name+".zip")
    target.write_bytes(raw)
    print(name, filename, len(raw), "bytes")
    return target
FACE_ZIP = upload_asset("face")
OUTFIT_ZIP = upload_asset("outfit")
HAIR_ZIP = upload_asset("hair")
ACCESSORY_ZIP = upload_asset("accessory") if WITH_ACCESSORY else None
print("모든 원화 ZIP 접수 완료. 다음 셀에서 실제 알파와 캔버스를 검사합니다.")

In [ ]:
#@title ③ See-through용 완성 캐릭터 원화 합성 및 시각 검수
from IPython.display import display
from PIL import Image as PILImage
ARTWORK_REPORT = composite_2d(FACE_ZIP, OUTFIT_ZIP, HAIR_ZIP,
                              WORK / "merged", ACCESSORY_ZIP)
with PILImage.open(ARTWORK_REPORT["composite_png"]) as image:
    display(image)
print(json.dumps(ARTWORK_REPORT, ensure_ascii=False, indent=2))
print("반드시 목/옷깃, 두상/헤어, 피부색, 시점/배치 정렬을 눈으로 확인하세요.")
print("오류가 있으면 원본 PNG를 수정한 뒤 다시 업로드합니다. GPU 분리는 아직 실행하지 않았습니다.")

In [ ]:
#@title ④ 실제 See-through + 공식 SemanticSAM PSD 분리
RUN_GPU_SEPARATION = False #@param {type:"boolean"}
FRAME = "upper" #@param ["upper","full"]
# Complete independently authored part_art ZIPs need NO GPU decomposition.
from tools.vts_authored_part_art import has_authored_parts
SOURCE_ZIPS = {"face": FACE_ZIP, "outfit": OUTFIT_ZIP, "hair": HAIR_ZIP}
if ACCESSORY_ZIP is not None:
    SOURCE_ZIPS["accessory"] = ACCESSORY_ZIP
PART_FLAGS = {name: has_authored_parts(path) for name, path in SOURCE_ZIPS.items()}
if any(PART_FLAGS.values()) and not all(PART_FLAGS.values()):
    raise RuntimeError("PARTIAL_AUTHORED_SOURCES: 모든 자산의 part_art가 필요합니다. 이미 그린 파츠를 잃지 않도록 중단합니다.")
USE_PRESERVED_PARTS = all(PART_FLAGS.values())
if not RUN_GPU_SEPARATION and not USE_PRESERVED_PARTS:
    for name in ("SEPARATION", "PACKAGE", "RIGGED", "NATIVE_QA"):
        globals().pop(name, None)
    print("독립 파츠가 없는 완성 원화입니다. 시각 검수 후 RUN_GPU_SEPARATION=True로 실행하세요.")
else:
    if "ARTWORK_REPORT" not in globals():
        raise RuntimeError("③ 합성/시각 검수 셀을 먼저 실행하세요.")
    SEPARATION = run_see_through(ARTWORK_REPORT, WORK / "result", FRAME)
    PACKAGE = Path(SEPARATION["package"])
    if not PACKAGE.is_file():
        raise RuntimeError("실제 PSD ZIP이 생성되지 않았습니다")
    print("PSD 작업 ZIP:", PACKAGE)
    print("원본 RGBA 파츠 보존" if SEPARATION.get("preserved_authored_parts") else "See-through GPU 분리", "경로 완료.")
    print("PSD 원화가 생성됐으며 실제 방송용 움직임·가림 품질은 별도 검증해야 합니다.")

In [ ]:
#@title ⑤ 선택적 PSD2Live 자동 리깅 (기본 비활성)
RIGGING = "artwork_only" #@param ["artwork_only","auto","auto_repair","modern"]
if RIGGING in ("auto", "auto_repair", "modern"):
    if "SEPARATION" not in globals() or "PACKAGE" not in globals() or not PACKAGE.is_file():
        raise RuntimeError("④에서 See-through PSD 생성과 ZIP 검증을 먼저 완료하세요")
    if RIGGING == "modern":
        # Official v3.6.0 checkout is independent from the pinned v3.3.1 bridge.
        from tools.vts_psd2live import rig_existing_artwork_zip
        RIGGED = rig_existing_artwork_zip(
            Path(SEPARATION["package"]), WORK / "rigging",
            edition="free", scope=FRAME, backend="modern")
    elif RIGGING == "auto_repair":
        from tools.vts_psd2live import rig_existing_artwork_zip
        RIGGED = rig_existing_artwork_zip(
            Path(SEPARATION["package"]), WORK / "rigging", edition="free",
            scope=FRAME, auto_repair=True)
    else:
        from tools.vts_see_through_only import rig_see_through_artwork
        RIGGED = rig_see_through_artwork(SEPARATION, WORK / "rigging", FRAME)
    PACKAGE = Path(RIGGED["package"])
    if not PACKAGE.is_file() or not RIGGED.get("moc3_generated"):
        raise RuntimeError("네이티브 PSD2Live 결과 파일이 없습니다")
    print("자동 리깅 MOC3/CMO3:", PACKAGE)
    print("파일 생성은 완료됐지만 VTube Studio 추적과 시각적 리깅 품질은 별도 검증해야 합니다.")
else:
    globals().pop("RIGGED", None)
    print("기본 PSD 원화 경로를 유지합니다. 자동 리깅이 필요할 때만 auto를 선택하세요.")


In [ ]:
#@title ⑥ 선택적 실제 MOC3 포즈 검사 (방송 앱 검증과 별개)
VERIFY_NATIVE_POSES = False #@param {type:"boolean"}
if VERIFY_NATIVE_POSES or RIGGING == "auto_repair":
    if "RIGGED" not in globals() or RIGGING not in ("auto", "auto_repair", "modern"):
        raise RuntimeError("⑤에서 PSD2Live 자동 리깅을 먼저 완료하세요")
    from zipfile import ZipFile, ZIP_DEFLATED
    from tools.vts_multiview_runtime import render_actual_native_moc3
    MODEL_JSON = Path(RIGGED["psd2live"]["model3_json"])
    NATIVE_QA_DIR = WORK / "native_moc3_pose"
    NATIVE_QA = (RIGGED["native_pose"] if RIGGING == "auto_repair"
                 else render_actual_native_moc3(MODEL_JSON, NATIVE_QA_DIR))
    if not NATIVE_QA.get("pose_activity_passed"):
        raise RuntimeError("네이티브 MOC3 동작 검사가 통과하지 못했습니다")
    with ZipFile(PACKAGE, "a", ZIP_DEFLATED) as archive:
        existing = set(archive.namelist())
        for path in sorted(NATIVE_QA_DIR.glob("*.png")) + [NATIVE_QA_DIR / "native_pose_report.json"]:
            if not path.is_file():
                continue
            target = "qa/native_moc3/" + path.name
            if target not in existing:
                archive.write(path, target)
    print("실제 MOC3 포즈 동작 검사 통과. QA 자료를 ZIP에 포함했습니다.")
    print("VTube Studio에서의 얼굴 트래킹 및 방송 품질은 별도로 확인해야 합니다.")
else:
    print("네이티브 MOC3 검사는 선택되지 않았습니다. MOC3 파일이 있어도 애니메이션 품질은 미검증입니다.")


In [ ]:
#@title ⑦ PSD 또는 자동 리깅 결과 ZIP 다운로드
from google.colab import files
if "PACKAGE" not in globals() or not PACKAGE.is_file():
    raise RuntimeError("④ GPU 분리와 PSD 포장까지 완료해야 다운로드할 수 있습니다")
files.download(str(PACKAGE))